# IsoKernel Pipeline Demonstration
This notebook demonstrates the execution of the IsoKernel knowledge engineering pipeline.

The pipeline comprises two major phases:
1. **Phase 1: Schema-Less Triple Extraction** - Using an LLM to extract Subject-Predicate-Object triples from raw text.
2. **Phase 2: Refinement & Logic Clustering** - Using graph topology (Louvain/Leiden) and semantic embeddings to cluster entities and reduce noise.


In [1]:
import os
from IPython.display import IFrame
import yaml

from src.core.models import DocumentSource
from src.orchestrator.pipeline import PipelineOrchestrator

# Ensure config points to correct input/output directories
config_path = "config.yaml"

print("Current configuration for the pipeline:")
with open(config_path, 'r') as f:
    print(f.read())


orchestrator = PipelineOrchestrator(config_path=config_path, domain="clinical child psychiatry")

Current configuration for the pipeline:
# IsoKernel Master Configuration

pipeline:
  run_phase_1: true
  run_phase_2: true
  
extraction:
  domain: "general knowledge"
  max_retries: 3
  
refinement:
  # Option A: Topology clustering
  community_detection: "louvain"  # or "leiden"
  
  # Option B: Embedding parameters
  use_embeddings: true
  embedding_model: "all-MiniLM-L6-v2"
  clustering_method: "agglomerative"  # or "hdbscan"
  similarity_threshold: 0.8  # Threshold for collapsing nodes using Agglomerative
  compression_mode: "isolated" # or "unified"
  compress_fields: ["subject", "object", "predicate"] # toggle any subset of valid fields
  hypernym_resolution: "most_frequent" # or "shortest_string"
  
output:
  logs_dir: "data/outputs/logs"
  schemas_dir: "data/outputs/schemas"
  graphs_dir: "data/outputs/graphs"



## Executing the Pipeline
Now we will run the orchestrator on our `input.txt` file. This text will be processed to extract triples and build the knowledge graph topology.

In [2]:
#input_text_file = "input.txt"

# Run the pipeline
#print(f"Executing pipeline on {input_text_file}...")
#orchestrator.run([input_text_file])

folder_path = "./data/inputs"

orchestrator.run(folder_path)
print("Pipeline execution complete!")


2026-04-10 16:20:38,370 | orchestrator | INFO | Starting IsoKernel Knowledge Graph Pipeline
2026-04-10 16:20:38,371 | orchestrator | INFO | Discovered 3 file(s) and 0 preloaded document(s) for processing.
2026-04-10 16:20:38,371 | phase1_extractor | INFO | Configuring local LLM provider via AsyncOpenAI API schema...
2026-04-10 16:20:38,524 | phase1_extractor | INFO | Initialized TripleExtractor for domain 'clinical child psychiatry' using model 'mistral-nemo:12b-instruct-2407-q4_K_M' via 'local'
2026-04-10 16:20:38,524 | orchestrator | INFO | Processing document ID: doc2.txt
2026-04-10 16:20:40,032 | orchestrator | INFO | Processing document ID: doc3.txt
2026-04-10 16:20:41,532 | orchestrator | INFO | Processing document ID: doc1.txt
2026-04-10 16:22:09,581 | orchestrator | INFO | Phase 1 completed. 30 total raw triples saved to data/outputs/schemas/phase1_raw_triples.json
2026-04-10 16:22:09,583 | embedding_service | INFO | Using localized model directory: /home/ai-admin/Desktop/Agent

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-04-10 16:22:12,205 | phase2_processor | INFO | Saving sleek graph visualization to data/outputs/graphs/01_initial_population.html
2026-04-10 16:22:12,216 | embedding_service | INFO | Running Semantic Compression (Option B)...
2026-04-10 16:22:12,216 | embedding_service | INFO | Computing embeddings for 28 unique nodes.
2026-04-10 16:22:12,427 | phase2_processor | INFO | Saving sleek graph visualization to data/outputs/graphs/02_semantic_compression.html
2026-04-10 16:22:12,436 | phase2_processor | INFO | Building Core Directed Graph...
2026-04-10 16:22:12,436 | phase2_processor | INFO | Graph initialized with 28 nodes and 28 edges.
2026-04-10 16:22:12,437 | phase2_processor | INFO | Running Community Detection (louvain)...
2026-04-10 16:22:12,438 | phase2_processor | INFO | Saving sleek graph visualization to data/outputs/graphs/03_final_graph_with_communities.html
2026-04-10 16:22:12,446 | phase2_processor | INFO | Exporting logic schemas to data/outputs/schemas/phase2_refined_cl

Pipeline execution complete!


## Visualizing the Results
The orchestrator outputs interactive network graphs as HTML files in the `graph_progressions` directory. We can visualize the final clustered community graph below.

In [3]:
# Display the generated graph in the notebook
html_path = "./data/outputs/graphs/03_final_graph_with_communities.html"

if os.path.exists(html_path):
    display(IFrame(src=html_path, width="100%", height="750px"))
else:
    print(f"Output graph not found at {html_path}. Please check the pipeline logs.")
